## Import Modules

In [ ]:
import xarray as xr
import pandas as pd
from ioos_qc.stores import PandasStore
from ioos_qc.streams import Config, PandasStream

In [ ]:
def apply_qc(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    # Setup the stream
    stream = PandasStream(df)
    # Run the tests
    results = stream.run(config)
    # Store the results in another DataFrame
    store = PandasStore(
        results,
        axes={
            't': 'time',
            'z': 'z',
            'y': 'lat',
            'x': 'lon'
        }
    )
    # Compute any aggregations
    store.compute_aggregate(name='rollup_qc')  # Appends to the results internally
    # Write only the test results to the store
    results_store = store.save(write_data=False, write_axes=False)
    # Append columns from qc results back into the data
    return pd.concat([df, results_store], axis=1)

## Tidy Table for QC

In [ ]:
starting = "2026-01-02 00:00:00".to_pydatetime()
ending = "2026-01-19 23:59:59".to_pydatetime()

In [ ]:
starting = datetime.combine(starting, time.min).isoformat(timespec='seconds')
ending = datetime.combine(ending, time.max).isoformat(timespec='seconds')

In [ ]:
# Needs to contain columns temperature, salinty, lat, lon, time
df =

## Define QC Configuration

In [ ]:
time_bounds_qc = f"""
    time:
        axds:
            valid_range_test:
                valid_span:
                    - {starting}
                    - {ending}"""

In [ ]:
temperature_qc = """
    temperature:
        qartod:
            gross_range_test:
                fail_span: [-2.5, 40]
    """

salinity_qc = """
    salinity:
        qartod:
            gross_range_test:
                fail_span: [0.0, 41.0]
    """

speed_qc = """
    lon:
        argo:
            speed_test:
                suspect_threshold: 8.0
                fail_threshold: 10.0
    """

location_qc = """
    lon:
        qartod:
            location_test:
                bbox: [-180, -90, 180, 90]
    """

In [ ]:
qcconfig = Config(time_bounds_qc)
qcconfig.add(Config(temperature_qc))
qcconfig.add(Config(salinity_qc))
qcconfig.add(Config(speed_qc))
qcconfig.add(Config(location_qc))

## Apply QC

In [ ]:
# Run the QC
full_df = apply_qc(df, qcconfig)